# Extract VPS35 variants 

`GP2 ❤️ Open Science 😍`

- **Project:** The VPS35 p.A320V variant segregates with Parkinson’s disease in a pesticide-exposed family

## Imports

In [ ]:
#!/usr/bin/env python3
import pandas as pd
import subprocess
from pathlib import Path

## Configuration

In [ ]:
PLINK2 = "/opt/workbench-tools/binaries/bin/plink2"

DATASETS_ROOT = Path("/path/to/gp2/release/11/wgs/deepvariant_joint_calling/plink")   
OUTPUT_ROOT   = Path("/oath/to/home/VPS35_candidate_variant")   
OUTPUT_ROOT.mkdir(exist_ok=True)

MANIFEST = Path("/path/to/gp2/release/11/clinical_data/master_key_release11_final_vwb.csv")   # must contain IID column

DATASET_NAMES = [
   "AAC","AFR","AJ","AMR","CAH","CAS","EAS","EUR","FIN","MDE","SAS"
]

VARIANT = {"chr": "16", "pos": 46674616, "ref": "G", "alt": "A"}

## Helper functions

In [ ]:
def find_variant_in_pvar(pvar_path, variant):
    """
    Return the variant ID in the .pvar file matching chr/pos.
    Alleles are NOT enforced (GBA1 often flips REF/ALT).
    """
    target_chr = variant["chr"]
    target_pos = int(variant["pos"])

    with open(pvar_path) as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            chrom, pos, vid, ref, alt = parts[:5]
            chrom_norm = chrom.replace("chr", "")

            if chrom_norm == target_chr and int(pos) == target_pos:
                print(f"  Found variant in {pvar_path.name}: {vid}")
                return vid

    return None


def run_plink_extract(plink2, pfile_prefix, variant_id, out_prefix):
    """Run plink2 extraction for a single variant."""
    snplist = out_prefix.with_suffix(".snplist")
    snplist.write_text(variant_id + "\n")

    cmd = [
        plink2,
        "--pfile", pfile_prefix,
        "--extract", str(snplist),
        "--recode", "A",
        "--out", str(out_prefix)
    ]
    subprocess.run(cmd, check=True)


def find_genotype_column(df, variant):
    """
    Find the genotype column in .raw by matching chr + pos.
    This is required because PLINK may rewrite variant IDs.
    """
    target_chr = variant["chr"]
    target_pos = str(variant["pos"])

    for col in df.columns:
        if ":" not in col:
            continue
        parts = col.split(":")
        if len(parts) < 2:
            continue

        chrom = parts[0].replace("chr", "")
        pos = parts[1]

        if chrom == target_chr and pos == target_pos:
            return col

    return None


## Variant extraction loop

In [ ]:
all_carriers = []

for ds in DATASET_NAMES:
    print(f"\nProcessing {ds}...")

    cohort_dir = DATASETS_ROOT / ds

    # Correct pfile prefix
    pfile_prefix = cohort_dir / f"chr16_{ds}_release11"

    pvar_path = pfile_prefix.with_suffix(".pvar")
    raw_prefix = OUTPUT_ROOT / f"{ds}_extract"

    # 1. Find variant ID in .pvar
    vid = find_variant_in_pvar(pvar_path, VARIANT)
    if vid is None:
        print(f"  Variant not found in {ds}")
        continue

    # 2. Extract variant using plink2
    run_plink_extract(PLINK2, pfile_prefix, vid, raw_prefix)

    raw_path = raw_prefix.with_suffix(".raw")
    if not raw_path.exists():
        print(f"  No .raw file produced for {ds}")
        continue

    # 3. Load raw file
    df = pd.read_csv(raw_path, sep=r"\s+", engine="python")
    print(f"  Raw columns: {list(df.columns)}")

    # 4. Detect actual genotype column
    geno_col = find_genotype_column(df, VARIANT)
    print(f"  Genotype column detected: {geno_col}")

    if geno_col is None:
        print(f"  No genotype column found for {ds}")
        continue

    # 5. Extract carriers
    geno = pd.to_numeric(df[geno_col], errors="coerce")
    
    # Only heterozygous carriers
    carriers = df[(geno == 1)].copy()
    
    print(f"  Het carriers found in {ds}: {carriers.shape[0]}")
    
    if carriers.empty:
        continue
    
    carriers["dataset"] = ds
    carriers["variant_id"] = geno_col
    carriers["genotype"] = geno[(geno == 1)]


    all_carriers.append(carriers)

## Merge all carriers

In [ ]:
if len(all_carriers) == 0:
    print("\nNo carriers found in any dataset.")
    exit()

merged = pd.concat(all_carriers, ignore_index=True)

## Annotate with manifest

In [ ]:
manifest = pd.read_csv(MANIFEST)

# Rename IID in carriers so it matches StudyID in manifest
merged = merged.rename(columns={"IID": "GP2ID"})

annotated = merged.merge(manifest, on="GP2ID", how="left")

## Subset final table to selected columns

In [ ]:
# Rename IID so it matches manifest
merged = merged.rename(columns={"IID": "GP2ID"})

# Merge with manifest
annotated = merged.merge(manifest, on="GP2ID", how="left")

# Choose the columns you want in the final output
final_cols = [
    "GP2ID",      
    "biological_sex_for_qc",           
    "study",          
    "race_for_qc",       
    "baseline_GP2_phenotype",   
    "age_at_sample_collection",
    "age_of_onset",
    "family_history_for_qc"
]

# Subset
final_table = annotated[final_cols]


## Save output

In [ ]:
final_path1 = OUTPUT_ROOT / "VPS35_variant_carriers_annotated.csv"
annotated.to_csv(final_path, index=False)
print(f"\nDone. Annotated carriers written to {final_path1}")


final_path2 = OUTPUT_ROOT / "VPS35_variant_carriers_annotated_columns_of_interest.csv"
final_table.to_csv(final_path, index=False)
print(f"\nDone. Variables of interest written to {final_path2}")
